In [9]:
import json
import re
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from extract.recover_lda_client_zip_codes import (
    get_client_html_file,
    get_client_zip_codes,
)

In [10]:
conn = sqlite3.connect("E://irs990_full.db")

In [11]:
# We only care about US clients as that is outside of the scope
# Zip codes are already collected for clients that are also registrants
query = """
SELECT
    raw_json ->> '$.filing_uuid' as filing_id,
    raw_json ->> '$.filing_type_display' as filing_type,
    raw_json ->> '$.filing_document_url' as filing_doc,
    raw_json ->> '$.filing_document_content_type' as filing_doc_type,
    raw_json ->> '$.client.id' as client_id,
    raw_json ->> '$.client.general_description' as client_general_description,
    raw_json ->> '$.client.client_self_select' as client_self_select,
    raw_json ->> '$.client.state' as client_state,
    raw_json ->> '$.client.country' as client_country
FROM lda_filings
WHERE
    raw_json ->> '$.client' is not null
    and raw_json ->> '$.client.client_self_select' <> 1
    and raw_json ->> '$.client.country' in ('US', null)
"""

In [12]:
filings = pd.read_sql(
    query,
    conn
)
print(len(filings))
filings.head()

79341


,filing_id,filing_type,filing_doc,filing_doc_type,client_id,client_general_description,client_self_select,client_state,client_country
0,7866327b-c892-4430-b9f0-1f0f679c58c6,Registration,https://lda.senate.gov/filings/public/filing/7...,text/html,58116,Emergency services dispatch center,0,IL,US
1,d81e9034-9f25-4dde-be8d-6d6d251370f1,Registration,https://lda.senate.gov/filings/public/filing/d...,text/html,54165,Connectivity provider,0,DC,US
2,c6bb3242-0e97-4a95-be5b-31f7ca998f88,Registration,https://lda.senate.gov/filings/public/filing/c...,text/html,54166,State Commission representing the California a...,0,CA,US
3,8d148bc8-9865-4583-87a6-403122a638d4,Registration,https://lda.senate.gov/filings/public/filing/8...,text/html,54168,"A cement and concrete technology company, offe...",0,TX,US
4,21fe6923-4997-4d99-b9b0-8bc775f0e98a,Registration,https://lda.senate.gov/filings/public/filing/2...,text/html,54169,Medical Marijuana Dispensary,0,OH,US


In [ ]:
client_zip = {}
for i in range(len(filings)):
    if i % 1000 == 0:
        print(f"{i:,} filings parsed")
    if i >= 20_000:
        break
    row = filings.iloc[i].to_dict()
    addresses = get_client_zip_codes(
        soup=get_client_html_file(row['filing_doc'])
    )
    
    if not addresses:
        client_zip[row['filing_id']] = {
            'client': row['client_id'],
            'address_1': None,
            'address_2': None
        }
        continue
        
    add_1, add_2 = addresses
    client_zip[row['filing_id']] = {
        'client': row['client_id'],
        # Need to convert to list if storing in a json file
        'address_1': [address_part for address_part in add_1],
        'address_2': [address_part for address_part in add_2]
    }

0 filings parsed
1,000 filings parsed
2,000 filings parsed
3,000 filings parsed
4,000 filings parsed
5,000 filings parsed
6,000 filings parsed
7,000 filings parsed
8,000 filings parsed
9,000 filings parsed
10,000 filings parsed
11,000 filings parsed
12,000 filings parsed
13,000 filings parsed
14,000 filings parsed


In [ ]:
client_zip

{'7866327b-c892-4430-b9f0-1f0f679c58c6': {'client': 58116,
  'address_1': ['Homewood', 'IL', '60430', 'USA'],
  'address_2': []}}

In [ ]:
with open('..data/client_zip.json', mode='w') as f:
    f.write(json.dumps(client_zip))